# MSCOCO_CLIPS1 INT8 Full Validation

Kiểm định toàn bộ **MS COCO 2014 Karpathy Validation** giữa ONNX FP32 và ứng viên FP32 encoder + dynamic INT8 decoder. Test được khóa và không đọc.

## Input bắt buộc
1. output `MOB_MSCOCO_CLIPS1_INT8_Preflight`;
2. output `MOB_MSCOCO_CLIPS1_ONNX_FP32`;
3. output `MOB_MSCOCO_CLIPS1_Entry`;
4. output `01_MSCOCO_Data_Contract`;
5. dataset `mscoco2014-1`;
6. dataset/package `pycocoevalcap-1.2` và `ONNX_Runtime_Python313_Offline`;

Khi resume, gắn thêm output gần nhất của chính notebook này. Runtime: **CPU / Accelerator None / Internet Off**. Có thể chạy song song 5 notebook trên các tài khoản/phiên Kaggle độc lập.


## Cell 1 — Authorization, integrity and runtime


In [1]:
import hashlib, importlib.util, json, math, os, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

CONFIG = {"model_id": "MSCOCO_CLIPS1", "dataset": "MS COCO 2014", "split": "MS COCO 2014 Karpathy Validation", "teacher": "CLIP ViT-B/32", "student": "MobileNetV3-Large", "method": "Direct Logit-KD", "seed": 123, "prefix": "mscoco_clips1", "preflight": "MOB_MSCOCO_CLIPS1_INT8_Preflight", "fp32": "MOB_MSCOCO_CLIPS1_ONNX_FP32", "entry": "MOB_MSCOCO_CLIPS1_Entry", "data_contract": "PHASE1_MSCOCO_DATA_CONTRACT_READY.json", "validation": "mscoco_validation.parquet", "validation_sha256": "d6749c77988f74a43176bcf48e807910dafeb640a90c121b90e1ac3714a9e351", "vocab": "word_vocabulary_minfreq5.json", "image_pattern": "COCO_val2014_{image_id:012d}.jpg", "expected_images": 5000, "expected_references": 25010}
EXPERIMENT_ID = f"MOB_{CONFIG['model_id']}_INT8_Full_Validation"
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KLCN_OUTPUT_ROOT", "/kaggle/working")) / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json_atomic(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    os.replace(temporary, path)

def named_candidates(name):
    return sorted((p for p in INPUT_ROOT.rglob(name) if p.is_file()), key=lambda p: (len(p.parts), len(str(p)), str(p)))

def one_named(name):
    matches = named_candidates(name)
    assert matches, ("Không tìm thấy input", name)
    hashes = {sha256(p) for p in matches}
    assert len(hashes) == 1, ("Có nhiều file cùng tên nhưng khác nội dung", name, [str(p) for p in matches])
    return matches[0]

def by_name_hash(name, expected):
    matches = [p for p in named_candidates(name) if sha256(p) == expected]
    assert matches, ("Không tìm thấy artifact đúng SHA-256", name, expected)
    return matches[0]

PREFLIGHT_PATH = one_named(CONFIG["preflight"] + "_RESULT.json")
FP32_PATH = one_named(CONFIG["fp32"] + "_COMPLETE.json")
ENTRY_PATH = one_named(CONFIG["entry"].removesuffix("_Entry") + "_ENTRY_LOCK.json")
DATA_CONTRACT_PATH = one_named(CONFIG["data_contract"])
VAL_PATH = one_named(CONFIG["validation"])
PREFLIGHT, FP32, ENTRY, DATA_CONTRACT = map(read_json, [PREFLIGHT_PATH, FP32_PATH, ENTRY_PATH, DATA_CONTRACT_PATH])

assert PREFLIGHT["status"] == "passed"
assert PREFLIGHT["model_id"] == CONFIG["model_id"]
assert PREFLIGHT["selected_candidate"] == "fp32_encoder_dynamic_int8_decoder"
assert PREFLIGHT["next_phase_authorized"] == EXPERIMENT_ID
assert PREFLIGHT["test_set_used"] is False
assert FP32["status"] == "complete" and FP32["model_id"] == CONFIG["model_id"]
assert ENTRY["status"] == "locked" and ENTRY["selected_seed"] == CONFIG["seed"]
assert DATA_CONTRACT["test_set_status"] == "sealed"

ENCODER_NAME = FP32["encoder"]["file"]
DECODER_FP32_NAME = FP32["decoder"]["file"]
DECODER_INT8_NAME = CONFIG["prefix"] + "_decoder_dynamic_int8.onnx"
VOCAB_NAME = CONFIG["vocab"]
ENCODER = by_name_hash(ENCODER_NAME, FP32["encoder"]["sha256"])
DECODER_FP32 = by_name_hash(DECODER_FP32_NAME, FP32["decoder"]["sha256"])
DECODER_INT8 = by_name_hash(DECODER_INT8_NAME, PREFLIGHT["artifacts"][DECODER_INT8_NAME]["sha256"])
VOCAB_PATH = by_name_hash(VOCAB_NAME, FP32["vocabulary"]["sha256"])
VOCAB = read_json(VOCAB_PATH)

assert sha256(VAL_PATH) == CONFIG["validation_sha256"]

if importlib.util.find_spec("onnxruntime") is None:
    wheels = sorted(INPUT_ROOT.rglob("onnxruntime*.whl"))
    assert wheels, "Hãy gắn ONNX_Runtime_Python313_Offline."
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", str(wheels[0])])
if importlib.util.find_spec("pycocoevalcap") is None:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl"))
    assert wheels, "Hãy gắn pycocoevalcap-1.2 offline."
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", str(wheels[0])])
assert shutil.which("java"), "METEOR chính thức yêu cầu Java runtime."

import onnxruntime as ort
opts = ort.SessionOptions()
opts.intra_op_num_threads = max(1, os.cpu_count() or 1)
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
providers = ["CPUExecutionProvider"]
ENC_SESSION = ort.InferenceSession(str(ENCODER), sess_options=opts, providers=providers)
FP32_SESSION = ort.InferenceSession(str(DECODER_FP32), sess_options=opts, providers=providers)
INT8_SESSION = ort.InferenceSession(str(DECODER_INT8), sess_options=opts, providers=providers)
print("Authorization         : PASSED")
print("Model / seed          :", CONFIG["model_id"], "/", CONFIG["seed"])
print("Candidate             : FP32 encoder + dynamic INT8 decoder")
print("Validation            : AUTHORIZED")
print("Held-out Test         : SEALED / NOT READ")
print("ONNX Runtime          :", ort.__version__)


Authorization         : PASSED
Model / seed          : MSCOCO_CLIPS1 / 123
Candidate             : FP32 encoder + dynamic INT8 decoder
Validation            : AUTHORIZED
Held-out Test         : SEALED / NOT READ
ONNX Runtime          : 1.28.0


## Cell 2 — Validation data and deterministic Beam-3


In [2]:
val_df = pd.read_parquet(VAL_PATH)
assert val_df["image_id"].nunique() == CONFIG["expected_images"]
assert len(val_df) == CONFIG["expected_references"]
VAL_REFERENCES, VAL_TOKENS = {}, {}
for image_id, group in val_df.groupby("image_id", sort=True):
    iid = int(image_id)
    VAL_REFERENCES[iid] = [str(x) for x in group.caption.tolist()]
    VAL_TOKENS[iid] = [list(map(int, x)) for x in group.token_ids]

def image_name(iid):
    return CONFIG["image_pattern"].format(image_id=int(iid))

needed = {image_name(iid) for iid in VAL_REFERENCES}
IMAGE_BY_NAME, duplicate_aliases = {}, 0
for path in INPUT_ROOT.rglob("*.jpg"):
    if path.name not in needed:
        continue
    old = IMAGE_BY_NAME.get(path.name)
    if old is None:
        IMAGE_BY_NAME[path.name] = path
    else:
        assert old.stat().st_size == path.stat().st_size, ("Ảnh trùng tên khác nội dung", path.name)
        duplicate_aliases += 1
        IMAGE_BY_NAME[path.name] = min([old, path], key=lambda p: (len(p.parts), len(str(p)), str(p)))
missing = sorted(needed - set(IMAGE_BY_NAME))
assert not missing, ("Thiếu ảnh Validation", len(missing), missing[:10])

MEAN = np.asarray(FP32["preprocessing"]["mean"], dtype=np.float32).reshape(1, 1, 3)
STD = np.asarray(FP32["preprocessing"]["std"], dtype=np.float32).reshape(1, 1, 3)
if "tokens" in VOCAB:
    TOKENS = list(VOCAB["tokens"])
else:
    mapping = VOCAB["id_to_token"]
    TOKENS = list(mapping) if isinstance(mapping, list) else [mapping[str(i)] for i in range(len(mapping))]
SPECIAL = {0, 1, 2, 3}

def resize_shorter_side(image, target=256):
    w, h = image.size
    size = (target, round(target * h / w)) if w <= h else (round(target * w / h), target)
    return image.resize(size, resample=Image.Resampling.BICUBIC)

def preprocess(path):
    with Image.open(path) as opened:
        image = resize_shorter_side(opened.convert("RGB"))
    w, h = image.size
    left, top = round((w - 224) / 2), round((h - 224) / 2)
    arr = np.asarray(image.crop((left, top, left + 224, top + 224)), dtype=np.float32) / 255.0
    arr = (arr - MEAN) / STD
    return np.transpose(arr, (2, 0, 1))[None].astype(np.float32)

def ids_to_caption(ids):
    words = []
    for token_id in ids:
        token_id = int(token_id)
        if token_id == 2: break
        if token_id not in SPECIAL: words.append(TOKENS[token_id])
    text = " ".join(words)
    for punctuation in [".", ",", "!", "?", ";", ":"]:
        text = text.replace(" " + punctuation, punctuation)
    return text.strip()

def log_softmax(values):
    values = np.asarray(values, dtype=np.float64)
    shifted = values - float(np.max(values))
    return shifted - math.log(float(np.exp(shifted).sum()))

ENC_IN, ENC_OUT = FP32["encoder"]["input_name"], FP32["encoder"]["output_name"]
DEC_INPUTS = FP32["decoder"]["input_names"]
DEC_BATCH = int(FP32["decoder"]["batch"])
MASK_DTYPE = np.bool_ if FP32["decoder"]["mask_dtype"] == "bool" else np.int64

def decoder_feed(token_buffer, memory, attention):
    return {DEC_INPUTS[0]: token_buffer, DEC_INPUTS[1]: memory, DEC_INPUTS[2]: attention}

def beam_score(beam, alpha=0.7):
    return beam[1] / (max(1, len(beam[0]) - 1) ** alpha)

def beam_decode(session, memory, beam_size=3, alpha=0.7):
    beams, calls = [([1], 0.0, False)], 0
    for _ in range(30):
        active = [b for b in beams if not b[2]]
        if not active: break
        candidates = [b for b in beams if b[2]]
        for start in range(0, len(active), DEC_BATCH):
            group = active[start:start + DEC_BATCH]
            tokens = np.zeros((DEC_BATCH, 31), dtype=np.int64)
            masks = np.zeros((DEC_BATCH, 31), dtype=MASK_DTYPE)
            memories = np.repeat(memory, DEC_BATCH, axis=0)
            for row, (ids, _, _) in enumerate(group):
                tokens[row, :len(ids)] = ids
                masks[row, :len(ids)] = True if MASK_DTYPE == np.bool_ else 1
            logits = session.run(["logits"], decoder_feed(tokens, memories, masks))[0]
            calls += 1
            for row, (ids, score, _) in enumerate(group):
                lp = log_softmax(logits[row, len(ids) - 1])
                lp[0] = lp[1] = lp[3] = -np.inf
                top = np.argpartition(lp, -beam_size)[-beam_size:]
                top = sorted(map(int, top), key=lambda idx: (-float(lp[idx]), idx))
                for idx in top:
                    candidates.append((ids + [idx], score + float(lp[idx]), idx == 2))
        beams = sorted(candidates, key=lambda b: (-beam_score(b, alpha), tuple(b[0])))[:beam_size]
    best = min(beams, key=lambda b: (-beam_score(b, alpha), tuple(b[0])))
    return best[0], calls

def teacher_ce(session, memory, ids):
    sequence = list(map(int, ids))[:31]
    assert sequence and sequence[0] == 1
    # Một caption MS COCO dài có EOS ở vị trí sau giới hạn 31 bước. Việc
    # chuẩn hóa này khớp hợp đồng decoder: giữ BOS và buộc EOS ở bước cuối.
    if 2 not in sequence:
        sequence[-1] = 2
    tokens = np.zeros((DEC_BATCH, 31), dtype=np.int64)
    masks = np.zeros((DEC_BATCH, 31), dtype=MASK_DTYPE)
    for row in range(DEC_BATCH):
        tokens[row, :len(sequence)] = sequence
        masks[row, :len(sequence)] = True if MASK_DTYPE == np.bool_ else 1
    memories = np.repeat(memory, DEC_BATCH, axis=0)
    logits = session.run(["logits"], decoder_feed(tokens, memories, masks))[0][0]
    losses = [-float(log_softmax(logits[pos])[sequence[pos + 1]]) for pos in range(len(sequence) - 1)]
    return sum(losses), len(losses)

print("Validation images     :", len(VAL_REFERENCES))
print("Validation references :", sum(map(len, VAL_REFERENCES.values())))
print("Vocabulary            :", len(TOKENS))
print("Nested aliases        :", duplicate_aliases, "(deduplicated)")
print("Beam / penalty        : 3 / 0.7")


Validation images     : 5000
Validation references : 25010
Vocabulary            : 10080
Nested aliases        : 0 (deduplicated)
Beam / penalty        : 3 / 0.7


## Cell 3 — Full Validation inference with exact resume


In [3]:
FP32_CAPTIONS_PATH = WORK_ROOT / "fp32_validation_captions.json"
INT8_CAPTIONS_PATH = WORK_ROOT / "mixed_int8_validation_captions.json"
PROGRESS_PATH = WORK_ROOT / f"{EXPERIMENT_ID}_PROGRESS.json"
FP32_CAPTIONS, INT8_CAPTIONS = {}, {}
CE = {"fp32_loss_sum": 0.0, "int8_loss_sum": 0.0, "token_count": 0}
TIMING = {"encoder_seconds": 0.0, "fp32_decoder_seconds": 0.0, "int8_decoder_seconds": 0.0,
          "fp32_decoder_calls": 0, "int8_decoder_calls": 0}

resume = sorted(INPUT_ROOT.rglob(f"{EXPERIMENT_ID}_PROGRESS.json"))
if resume:
    assert len(resume) == 1, resume
    progress = read_json(resume[0])
    assert progress["preflight_sha256"] == sha256(PREFLIGHT_PATH)
    assert progress["validation_sha256"] == sha256(VAL_PATH)
    parent = resume[0].parent
    FP32_CAPTIONS = {int(k): v for k, v in read_json(parent / FP32_CAPTIONS_PATH.name).items()}
    INT8_CAPTIONS = {int(k): v for k, v in read_json(parent / INT8_CAPTIONS_PATH.name).items()}
    CE, TIMING = progress["ce"], progress["timing"]
    assert set(FP32_CAPTIONS) == set(INT8_CAPTIONS)
    print("Exact resume          :", len(FP32_CAPTIONS), "/", len(VAL_REFERENCES))

def save_progress():
    write_json_atomic(FP32_CAPTIONS_PATH, {str(k): FP32_CAPTIONS[k] for k in sorted(FP32_CAPTIONS)})
    write_json_atomic(INT8_CAPTIONS_PATH, {str(k): INT8_CAPTIONS[k] for k in sorted(INT8_CAPTIONS)})
    write_json_atomic(PROGRESS_PATH, {
        "schema_version": "mobile_int8_validation_progress_v1", "experiment_id": EXPERIMENT_ID,
        "completed_images": len(FP32_CAPTIONS), "total_images": len(VAL_REFERENCES),
        "preflight_sha256": sha256(PREFLIGHT_PATH), "validation_sha256": sha256(VAL_PATH),
        "ce": CE, "timing": TIMING, "validation_data_read": True, "test_set_used": False})

SESSION_BUDGET_SECONDS = int(os.environ.get("KLCN_SESSION_BUDGET_SECONDS", "35000"))
started_run = time.perf_counter()
for image_id in sorted(VAL_REFERENCES):
    if image_id in FP32_CAPTIONS: continue
    image = preprocess(IMAGE_BY_NAME[image_name(image_id)])
    t0 = time.perf_counter(); memory = ENC_SESSION.run([ENC_OUT], {ENC_IN: image})[0]
    TIMING["encoder_seconds"] += time.perf_counter() - t0
    assert memory.shape == (1, 49, 256) and np.isfinite(memory).all()
    t0 = time.perf_counter(); ids32, calls32 = beam_decode(FP32_SESSION, memory)
    loss32, count32 = teacher_ce(FP32_SESSION, memory, VAL_TOKENS[image_id][0])
    TIMING["fp32_decoder_seconds"] += time.perf_counter() - t0; TIMING["fp32_decoder_calls"] += calls32 + 1
    t0 = time.perf_counter(); ids8, calls8 = beam_decode(INT8_SESSION, memory)
    loss8, count8 = teacher_ce(INT8_SESSION, memory, VAL_TOKENS[image_id][0])
    TIMING["int8_decoder_seconds"] += time.perf_counter() - t0; TIMING["int8_decoder_calls"] += calls8 + 1
    assert count32 == count8
    FP32_CAPTIONS[image_id], INT8_CAPTIONS[image_id] = ids_to_caption(ids32), ids_to_caption(ids8)
    CE["fp32_loss_sum"] += loss32; CE["int8_loss_sum"] += loss8; CE["token_count"] += count32
    done = len(FP32_CAPTIONS)
    if done % 10 == 0:
        print(f"Validation {done:04d}/{len(VAL_REFERENCES)} | image={image_id} | elapsed={(time.perf_counter()-started_run)/60:.1f} min", flush=True)
    if done % 25 == 0: save_progress()
    if time.perf_counter() - started_run >= SESSION_BUDGET_SECONDS:
        print("Session budget reached; saving exact resume state.", flush=True); break
save_progress()
INFERENCE_COMPLETE = len(FP32_CAPTIONS) == len(VAL_REFERENCES)
print("Inference status      :", "COMPLETE" if INFERENCE_COMPLETE else "RESUME REQUIRED")
print("Completed             :", len(FP32_CAPTIONS), "/", len(VAL_REFERENCES))
print("Held-out Test         : SEALED / NOT READ")


Validation 0010/5000 | image=1369 | elapsed=0.1 min
Validation 0020/5000 | image=2255 | elapsed=0.3 min
Validation 0030/5000 | image=4011 | elapsed=0.4 min
Validation 0040/5000 | image=4840 | elapsed=0.6 min
Validation 0050/5000 | image=5728 | elapsed=0.8 min
Validation 0060/5000 | image=6810 | elapsed=0.9 min
Validation 0070/5000 | image=8787 | elapsed=1.1 min
Validation 0080/5000 | image=9908 | elapsed=1.2 min
Validation 0090/5000 | image=11320 | elapsed=1.4 min
Validation 0100/5000 | image=12551 | elapsed=1.5 min
Validation 0110/5000 | image=13867 | elapsed=1.6 min
Validation 0120/5000 | image=15067 | elapsed=1.8 min
Validation 0130/5000 | image=16046 | elapsed=1.9 min
Validation 0140/5000 | image=17244 | elapsed=2.1 min
Validation 0150/5000 | image=18046 | elapsed=2.2 min
Validation 0160/5000 | image=19109 | elapsed=2.4 min
Validation 0170/5000 | image=20912 | elapsed=2.6 min
Validation 0180/5000 | image=21919 | elapsed=2.7 min
Validation 0190/5000 | image=22446 | elapsed=2.9 min
V

## Cell 4 — Official metrics, gates and manifest


In [4]:
if not INFERENCE_COMPLETE:
    write_json_atomic(WORK_ROOT / f"{EXPERIMENT_ID}_RESUME_REQUIRED.json", {
        "status": "resume_required", "experiment_id": EXPERIMENT_ID,
        "completed_images": len(FP32_CAPTIONS), "total_images": len(VAL_REFERENCES),
        "attach_latest_output_to_resume": True, "test_set_used": False})
    print("Metrics deferred until all Validation images are complete.")
else:
    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    def official_metrics(captions):
        refs = {int(k): [{"caption": x} for x in v] for k, v in VAL_REFERENCES.items()}
        hyps = {int(k): [{"caption": captions[int(k)]}] for k in VAL_REFERENCES}
        tokenizer = PTBTokenizer(); refs, hyps = tokenizer.tokenize(refs), tokenizer.tokenize(hyps)
        bleu, _ = Bleu(4).compute_score(refs, hyps)
        meteor = Meteor()
        try: meteor_value, _ = meteor.compute_score(refs, hyps)
        finally:
            close = getattr(meteor, "close", None)
            if callable(close): close()
        rouge, _ = Rouge().compute_score(refs, hyps); cider, _ = Cider().compute_score(refs, hyps)
        return {"BLEU_1": float(bleu[0]), "BLEU_4": float(bleu[3]), "METEOR": float(meteor_value),
                "ROUGE_L": float(rouge), "CIDEr_D": float(cider)}
    FP32_METRICS, INT8_METRICS = official_metrics(FP32_CAPTIONS), official_metrics(INT8_CAPTIONS)
    FP32_METRICS["ce_loss"] = CE["fp32_loss_sum"] / CE["token_count"]
    INT8_METRICS["ce_loss"] = CE["int8_loss_sum"] / CE["token_count"]
    limits = {"BLEU_1": .005, "BLEU_4": .005, "METEOR": .005, "ROUGE_L": .005, "CIDEr_D": .010}
    quality = {m: {"fp32": FP32_METRICS[m], "mixed_int8": INT8_METRICS[m],
                   "delta": INT8_METRICS[m]-FP32_METRICS[m], "maximum_allowed_drop": lim,
                   "passed": INT8_METRICS[m]-FP32_METRICS[m] >= -lim} for m, lim in limits.items()}
    ce_delta = INT8_METRICS["ce_loss"] - FP32_METRICS["ce_loss"]
    quality["ce_loss"] = {"fp32": FP32_METRICS["ce_loss"], "mixed_int8": INT8_METRICS["ce_loss"],
                          "increase": ce_delta, "maximum_allowed_increase": .05, "passed": ce_delta <= .05}
    reproduction = {"locked_pytorch_CIDEr_D": ENTRY["kd_cider_d"], "onnx_fp32_CIDEr_D": FP32_METRICS["CIDEr_D"],
                    "absolute_difference": abs(FP32_METRICS["CIDEr_D"]-ENTRY["kd_cider_d"]), "tolerance": .01}
    reproduction["passed"] = reproduction["absolute_difference"] <= reproduction["tolerance"]
    agreement = sum(FP32_CAPTIONS[k] == INT8_CAPTIONS[k] for k in FP32_CAPTIONS) / len(FP32_CAPTIONS)
    size_passed = PREFLIGHT["selected_size"]["reduction_fraction"] >= .40
    quality_passed = all(x["passed"] for x in quality.values())
    scientific_passed = reproduction["passed"] and quality_passed and size_passed
    status = "passed" if scientific_passed and agreement >= .90 else ("passed_with_caption_variation_review_required" if scientific_passed else "failed")
    result = {
        "schema_version": "mobile_int8_full_validation_v1", "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": status, "experiment_id": EXPERIMENT_ID, "model_id": CONFIG["model_id"],
        "teacher": CONFIG["teacher"], "student": CONFIG["student"], "dataset": CONFIG["dataset"],
        "method": CONFIG["method"], "seed": CONFIG["seed"], "evaluation_split": CONFIG["split"],
        "validation_images": len(VAL_REFERENCES), "validation_references": sum(map(len, VAL_REFERENCES.values())),
        "decoding": {"beam_size": 3, "length_penalty": .7, "max_steps": 30},
        "metrics": {"onnx_fp32": FP32_METRICS, "mixed_int8": INT8_METRICS,
                    "mixed_minus_fp32": {m: INT8_METRICS[m]-FP32_METRICS[m] for m in FP32_METRICS},
                    "x100": {"onnx_fp32": {m: round(v*100,2) for m,v in FP32_METRICS.items() if m != "ce_loss"},
                              "mixed_int8": {m: round(v*100,2) for m,v in INT8_METRICS.items() if m != "ce_loss"}}},
        "gates": {"fp32_reproduction": reproduction, "mixed_int8_quality": quality,
                  "caption_agreement": {"value": agreement, "minimum_diagnostic": .90, "passed": agreement >= .90},
                  "size": {"reduction_fraction": PREFLIGHT["selected_size"]["reduction_fraction"], "minimum": .40, "passed": size_passed}},
        "source_integrity": {"preflight_sha256": sha256(PREFLIGHT_PATH), "fp32_contract_sha256": sha256(FP32_PATH),
                             "entry_sha256": sha256(ENTRY_PATH), "validation_sha256": sha256(VAL_PATH),
                             "encoder_sha256": sha256(ENCODER), "decoder_fp32_sha256": sha256(DECODER_FP32),
                             "decoder_int8_sha256": sha256(DECODER_INT8)},
        "timing": TIMING, "validation_data_read": True, "test_set_status": "sealed", "test_set_used": False,
        "next_phase_authorized": (f"MOB_{CONFIG['model_id']}_Android_Lock" if scientific_passed else "STOP_REVIEW_INT8_QUALITY_FAILURE")}
    result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"; write_json_atomic(result_path, result)
    rows = [{"image_id": iid, "fp32_caption": FP32_CAPTIONS[iid], "mixed_int8_caption": INT8_CAPTIONS[iid],
             "exact_match": FP32_CAPTIONS[iid] == INT8_CAPTIONS[iid], "references": " || ".join(VAL_REFERENCES[iid])}
            for iid in sorted(VAL_REFERENCES)]
    csv_path = WORK_ROOT / f"{EXPERIMENT_ID}_CAPTIONS.csv"; pd.DataFrame(rows).to_csv(csv_path, index=False, encoding="utf-8")
    artifacts = [FP32_CAPTIONS_PATH, INT8_CAPTIONS_PATH, PROGRESS_PATH, result_path, csv_path]
    write_json_atomic(WORK_ROOT / "MANIFEST.json", {"schema_version": "mobile_int8_validation_manifest_v1",
        "experiment_id": EXPERIMENT_ID, "status": "complete", "artifacts": [
            {"relative_path": p.name, "bytes": p.stat().st_size, "sha256": sha256(p)} for p in artifacts],
        "validation_data_read": True, "test_set_used": False})
    print("Status               :", status.upper())
    print("FP32 / INT8 CIDEr-D  :", FP32_METRICS["CIDEr_D"], "/", INT8_METRICS["CIDEr_D"])
    print("Caption agreement    :", agreement)
    print("Size reduction       :", PREFLIGHT["selected_size"]["reduction_fraction"])
    print("Next                 :", result["next_phase_authorized"])
    assert scientific_passed, result["gates"]


PTBTokenizer tokenized 307820 tokens at 543743.18 tokens per second.
PTBTokenizer tokenized 53708 tokens at 244108.60 tokens per second.


{'testlen': 48709, 'reflen': 48192, 'guess': [48709, 43709, 38709, 33709], 'correct': [35248, 18703, 9155, 4520]}
ratio: 1.0107279216467255


PTBTokenizer tokenized 307820 tokens at 594986.00 tokens per second.
PTBTokenizer tokenized 53695 tokens at 271903.40 tokens per second.


{'testlen': 48696, 'reflen': 48192, 'guess': [48696, 43696, 38696, 33696], 'correct': [35270, 18747, 9228, 4586]}
ratio: 1.0104581673306563
Status               : PASSED_WITH_CAPTION_VARIATION_REVIEW_REQUIRED
FP32 / INT8 CIDEr-D  : 1.004451013192079 / 1.0073059607445285
Caption agreement    : 0.8738
Size reduction       : 0.5358023198204133
Next                 : MOB_MSCOCO_CLIPS1_Android_Lock
